In [1]:
import pandas as pd 
import numpy as np
import pickle
import os
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
from rdkit.ML.Descriptors  import MoleculeDescriptors
import warnings
warnings.filterwarnings('ignore')

project_path = r"C:\Users\Dell\Documents\SK\MyProject\Task1"

# Load Models 

with open(project_path + r'\models\rf_classifier.pkl', 'rb') as f:
    rf_model = pickle.load(f)
with open(project_path + r'\models\rf_regressor.pkl', 'rb') as f:
    rf_reg = pickle.load(f)
with open(project_path + r'\models\clean_descriptors.pkl', 'rb') as f:
    clean_descriptors = pickle.load(f)
with open(project_path + r'\models\descriptor_calculator.pkl', 'rb') as f:
    calculator = pickle.load(f)

print('env ready ')
print("models loaded Successfully")
print(f"clean descriptors : {len(clean_descriptors)}")


env ready 
models loaded Successfully
clean descriptors : 179


## Library Path

In [2]:
from rdkit import Chem
import os

sdf_path = r"D:\JK\CheB\Remaining Libraries to run\ibs2025mar_nc.sdf"

print(f"File size: {os.path.getsize(sdf_path)/1024/1024:.1f} MB")
print("Loading SDF file...")

supplier = Chem.SDMolSupplier(sdf_path)
total = len(supplier)
print(f"Total molecules: {total:,}")

# Preview first molecule
mol = supplier[0]
if mol is not None:
    props = mol.GetPropsAsDict()
    print(f"\nAvailable properties:")
    for key, val in list(props.items()):
        print(f"  {key:30s}: {val}")
    print(f"\nSMILES: {Chem.MolToSmiles(mol)[:80]}...")

File size: 231.2 MB
Loading SDF file...
Total molecules: 68,988

Available properties:
  id                            : STOCK1N-00002
  comment                       : 2 Isomers (12:1)
  index                         : RDNC
  donors                        : 2
  acceptors                     : 7
  rotatable                     : 8
  clogp                         : 2.656
  rings                         : 4
  tpsa                          : 85.89
  iupac_name                    : (S)-N-(10-((4-fluorobenzyl)amino)-1,2,3-trimethoxy-9-oxo-5,6,7,9-tetrahydrobenzo[a]heptalen-7-yl)acetamide

SMILES: COc1cc2c(c(OC)c1OC)-c1ccc(NCc3ccc(F)cc3)c(=O)cc1[C@@H](NC(C)=O)CC2...


## Screening

In [3]:
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
from rdkit.Chem import rdMolDescriptors
import pandas as pd
import numpy as np
import time
from collections import Counter

sdf_path = r"D:\JK\CheB\Remaining Libraries to run\ibs2025mar_nc.sdf"

# ZBG patterns
zbg_patterns = {
    'Hydroxamic acid' : Chem.MolFromSmarts('[CX3](=O)[NX3][OH]'),
    'Carboxylic acid' : Chem.MolFromSmarts('[CX3](=O)[OH]'),
    'Thiol'           : Chem.MolFromSmarts('[SX2H]'),
    'Phosphonate'     : Chem.MolFromSmarts('[PX4](=O)([OH])[OH]'),
    'Sulfonamide'     : Chem.MolFromSmarts('[SX4](=O)(=O)[NX3]'),
    'Catechol'        : Chem.MolFromSmarts('c1ccc(O)c(O)c1'),
    'N-hydroxy'       : Chem.MolFromSmarts('[NX3][OH]'),
    'Hydroxypyridine' : Chem.MolFromSmarts('n1ccccc1O'),
    'Beta-lactam'     : Chem.MolFromSmarts('[NX3]1[CX4][CX3](=O)1'),
}

print("Step 1 — Filtering IBS library...")
supplier = Chem.SDMolSupplier(sdf_path)

filtered_data = []
t0 = time.time()

for i, mol in enumerate(supplier):
    if mol is None:
        continue
    try:
        smiles = Chem.MolToSmiles(mol)
        props  = mol.GetPropsAsDict()

        comp_id  = props.get('id', f'IBS_{i}')
        comment  = props.get('comment', '')
        iupac    = props.get('iupac_name', '')
        mw       = Descriptors.MolWt(mol)
        logp     = float(props.get('clogp',
                         Descriptors.MolLogP(mol)))
        tpsa     = float(props.get('tpsa', 0))
        hbd      = int(props.get('donors',
                       rdMolDescriptors.CalcNumHBD(mol)))
        hba      = int(props.get('acceptors',
                       rdMolDescriptors.CalcNumHBA(mol)))
        rings    = int(props.get('rings', 0))

        # ZBG filter
        found_zbgs = []
        for zbg_name, pattern in zbg_patterns.items():
            if pattern and mol.HasSubstructMatch(pattern):
                found_zbgs.append(zbg_name)

        if not found_zbgs:
            continue

        filtered_data.append({
            'id'     : comp_id,
            'iupac'  : iupac,
            'comment': comment,
            'smiles' : smiles,
            'MW'     : round(mw, 2),
            'LogP'   : round(logp, 2),
            'TPSA'   : round(tpsa, 2),
            'HBD'    : hbd,
            'HBA'    : hba,
            'rings'  : rings,
            'ZBG'    : ', '.join(found_zbgs)
        })
    except:
        continue

    if (i+1) % 10000 == 0:
        elapsed = round((time.time()-t0)/60, 1)
        print(f"  Processed: {i+1:,} | "
              f"ZBG pass: {len(filtered_data):,} | "
              f"Time: {elapsed} mins")

filtered_df = pd.DataFrame(filtered_data)
print(f"\nTotal processed : 68,988")
print(f"ZBG passed      : {len(filtered_df):,}")
print(f"\nZBG breakdown:")
all_zbgs = []
for zbg in filtered_df['ZBG']:
    all_zbgs.extend(zbg.split(', '))
for zbg, count in Counter(all_zbgs).most_common():
    print(f"  {zbg:25s}: {count:,}")

# Step 2 — Screen with RF model
print("\nStep 2 — Screening with RF model...")
descriptor_names = [d[0] for d in Descriptors.descList]

desc_list, fp_list, valid_idx = [], [], []

for i, smiles in enumerate(filtered_df['smiles']):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is not None:
        descs = calculator.CalcDescriptors(mol)
        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol, radius=2, nBits=2048)
        desc_list.append(descs)
        fp_list.append(list(fp))
        valid_idx.append(i)
    if (i+1) % 2000 == 0:
        print(f"  {i+1:,}/{len(filtered_df):,}...",
              end='\r')

desc_df = pd.DataFrame(desc_list, columns=descriptor_names)
fp_df   = pd.DataFrame(
    fp_list,
    columns=[f'Morgan_{i}' for i in range(2048)]
)

X_desc = desc_df[clean_descriptors].copy()
X_desc = X_desc.replace([np.inf, -np.inf], np.nan)
X_desc = X_desc.fillna(X_desc.median())
X_desc = X_desc.clip(
    -np.finfo(np.float32).max,
     np.finfo(np.float32).max
).astype(np.float32)

X_comb = pd.concat([
    X_desc.reset_index(drop=True),
    fp_df.reset_index(drop=True)
], axis=1)
X_comb = X_comb.replace([np.inf, -np.inf], np.nan)
X_comb = X_comb.fillna(X_comb.median())
X_comb = X_comb.clip(
    -np.finfo(np.float32).max,
     np.finfo(np.float32).max
).astype(np.float32)

activity_pred = rf_model.predict(X_desc)
activity_prob = rf_model.predict_proba(X_desc)[:, 1]
pic50_pred    = rf_reg.predict(X_comb)

results = filtered_df.iloc[
    valid_idx
].reset_index(drop=True).copy()
results['predicted_activity'] = activity_pred
results['active_probability'] = activity_prob.round(4)
results['predicted_pIC50']    = pic50_pred.round(3)

actives = results[
    results['predicted_activity'] == 1
].sort_values(
    'active_probability', ascending=False
).reset_index(drop=True)

high_conf = actives[
    actives['active_probability'] >= 0.85
].copy()

# Save
actives.to_csv(
    project_path + r'\ibs_all_hits.csv', index=False
)
high_conf.to_csv(
    project_path + r'\glide_docking\ibs_docking.csv',
    index=False
)

elapsed = round((time.time()-t0)/60, 2)
print(f"\n{'='*55}")
print(f"IBS SCREENING COMPLETE")
print(f"{'='*55}")
print(f"ZBG compounds    : {len(filtered_df):,}")
print(f"Valid molecules  : {len(valid_idx):,}")
print(f"Actives          : {len(actives):,}")
print(f"High conf (≥0.85): {len(high_conf):,}")
print(f"Time             : {elapsed} mins")
print(f"\nTop 20 High Confidence Hits:")
print(high_conf.head(20)[[
    'id', 'ZBG', 'MW', 'LogP',
    'active_probability', 'predicted_pIC50'
]].to_string(index=False))

Step 1 — Filtering IBS library...


[12:15:31] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 4 ignored
[12:15:31] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 4 ignored
[12:15:32] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 15 ignored.
[12:15:32] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 15 ignored.
[12:15:32] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 9 ignored.
[12:15:32] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 9 ignored.
[12:15:33] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 2 ignored
[12:15:33] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 2 ignored
[12:15:33] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 2 ignored
[12:15:33] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 2 

  Processed: 10,000 | ZBG pass: 3,009 | Time: 0.3 mins


[12:15:45] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 3 ignored
[12:15:45] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 3 ignored
[12:15:45] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 8 ignored
[12:15:45] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 8 ignored
[12:15:45] Both bonds on one end of an atropisomer are on the same side - atoms is : 12
[12:15:45] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 2 ignored.
[12:15:45] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 2 ignored.
[12:15:45] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 5 ignored
[12:15:45] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 5 ignored
[12:15:45] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 5 ignored
[12:15:45] Warning: c


Total processed : 68,988
ZBG passed      : 27,042

ZBG breakdown:
  Catechol                 : 17,640
  Carboxylic acid          : 9,720
  Sulfonamide              : 918
  Hydroxypyridine          : 244
  Phosphonate              : 50
  Thiol                    : 43
  N-hydroxy                : 26
  Hydroxamic acid          : 15

Step 2 — Screening with RF model...


...

  24,000/27,042...

[12:25:47] DEPRECATION WARNING: please use MorganGenerator
[12:25:47] DEPRECATION WARNING: please use MorganGenerator
[12:25:48] DEPRECATION WARNING: please use MorganGenerator
[12:25:48] DEPRECATION WARNING: please use MorganGenerator
[12:25:48] DEPRECATION WARNING: please use MorganGenerator
[12:25:48] DEPRECATION WARNING: please use MorganGenerator
[12:25:48] DEPRECATION WARNING: please use MorganGenerator
[12:25:48] DEPRECATION WARNING: please use MorganGenerator
[12:25:48] DEPRECATION WARNING: please use MorganGenerator
[12:25:48] DEPRECATION WARNING: please use MorganGenerator
[12:25:48] DEPRECATION WARNING: please use MorganGenerator
[12:25:48] DEPRECATION WARNING: please use MorganGenerator
[12:25:48] DEPRECATION WARNING: please use MorganGenerator
[12:25:48] DEPRECATION WARNING: please use MorganGenerator
[12:25:48] DEPRECATION WARNING: please use MorganGenerator
[12:25:48] DEPRECATION WARNING: please use MorganGenerator
[12:25:48] DEPRECATION WARNING: please use MorganGenerat

  26,000/27,042...

[12:26:36] DEPRECATION WARNING: please use MorganGenerator
[12:26:36] DEPRECATION WARNING: please use MorganGenerator
[12:26:36] DEPRECATION WARNING: please use MorganGenerator
[12:26:36] DEPRECATION WARNING: please use MorganGenerator
[12:26:36] DEPRECATION WARNING: please use MorganGenerator
[12:26:36] DEPRECATION WARNING: please use MorganGenerator
[12:26:36] DEPRECATION WARNING: please use MorganGenerator
[12:26:36] DEPRECATION WARNING: please use MorganGenerator
[12:26:36] DEPRECATION WARNING: please use MorganGenerator
[12:26:36] DEPRECATION WARNING: please use MorganGenerator
[12:26:36] DEPRECATION WARNING: please use MorganGenerator
[12:26:36] DEPRECATION WARNING: please use MorganGenerator
[12:26:36] DEPRECATION WARNING: please use MorganGenerator
[12:26:36] DEPRECATION WARNING: please use MorganGenerator
[12:26:36] DEPRECATION WARNING: please use MorganGenerator
[12:26:36] DEPRECATION WARNING: please use MorganGenerator
[12:26:36] DEPRECATION WARNING: please use MorganGenerat


IBS SCREENING COMPLETE
ZBG compounds    : 27,042
Valid molecules  : 27,042
Actives          : 23,746
High conf (≥0.85): 935
Time             : 12.04 mins

Top 20 High Confidence Hits:
           id             ZBG     MW  LogP  active_probability  predicted_pIC50
STOCK1N-63806 Carboxylic acid 478.51  3.14                0.95            8.253
STOCK1N-66266 Carboxylic acid 450.45  1.96                0.95            8.083
STOCK1N-57232 Carboxylic acid 450.45  1.96                0.95            8.083
STOCK1N-60923 Carboxylic acid 549.58  2.63                0.95            8.118
STOCK1N-65839 Carboxylic acid 450.45  1.96                0.95            8.083
STOCK1N-66157 Carboxylic acid 551.63  2.61                0.94            8.289
STOCK1N-09833 Carboxylic acid 716.77 -0.06                0.94            8.190
STOCK1N-62225 Carboxylic acid 517.54  2.98                0.94            8.268
STOCK1N-59530 Carboxylic acid 521.53  1.71                0.94            8.158
STOCK1N-57987 C

## SDF file creation

In [4]:
from rdkit import Chem
from rdkit.Chem import AllChem
import pandas as pd

# Load high confidence hits
high_conf = pd.read_csv(
    project_path + r'\glide_docking\ibs_docking.csv'
)

print(f"Total compounds to convert: {len(high_conf)}")

# Write SDF
sdf_path = project_path + \
    r'\glide_docking\ibs_highconf_935.sdf'

writer = Chem.SDWriter(sdf_path)
success = 0
failed  = 0

for _, row in high_conf.iterrows():
    try:
        mol = Chem.MolFromSmiles(str(row['smiles']))
        if mol is None:
            failed += 1
            continue

        # Add hydrogens and generate 3D
        mol = Chem.AddHs(mol)
        result = AllChem.EmbedMolecule(
            mol, AllChem.ETKDGv3()
        )

        if result == 0:
            AllChem.MMFFOptimizeMolecule(mol)
        else:
            # Fall back to 2D if 3D fails
            mol = Chem.RemoveHs(mol)
            AllChem.Compute2DCoords(mol)

        # Set properties
        mol.SetProp('_Name',            str(row['id']))
        mol.SetProp('ZBG',              str(row['ZBG']))
        mol.SetProp('MW',               str(row['MW']))
        mol.SetProp('LogP',             str(row['LogP']))
        mol.SetProp('active_probability',
                    str(row['active_probability']))
        mol.SetProp('predicted_pIC50',
                    str(row['predicted_pIC50']))

        writer.write(mol)
        success += 1

        if success % 100 == 0:
            print(f"  Written: {success}...", end='\r')

    except:
        failed += 1

writer.close()

print(f"\n{'='*50}")
print(f"SDF FILE CREATED ✅")
print(f"{'='*50}")
print(f"Successfully written : {success}")
print(f"Failed               : {failed}")
print(f"Saved to             : {sdf_path}")

Total compounds to convert: 935
  Written: 900...
SDF FILE CREATED ✅
Successfully written : 935
Failed               : 0
Saved to             : C:\Users\Dell\Documents\SK\MyProject\Task1\glide_docking\ibs_highconf_935.sdf


## MEGx Import

In [5]:
from rdkit import Chem
import os

sdf_path = r"D:\JK\CheB\Remaining Libraries to run\MEGx_Release_2025_09_03_All_6539_cpds.sdf"

print(f"File size: {os.path.getsize(sdf_path)/1024/1024:.1f} MB")
print("Loading SDF file...")

supplier = Chem.SDMolSupplier(sdf_path)
total = len(supplier)
print(f"Total molecules: {total:,}")

# Preview first molecule
mol = supplier[0]
if mol is not None:
    props = mol.GetPropsAsDict()
    print(f"\nAvailable properties:")
    for key, val in list(props.items()):
        print(f"  {key:30s}: {val}")
    print(f"\nSMILES: {Chem.MolToSmiles(mol)[:80]}...")

File size: 25.0 MB
Loading SDF file...
Total molecules: 6,539

Available properties:
  Compound ID                   : NP-000001
  CAS                           : 526-31-8
  Release Date                  : 3 September 2025
  Collection                    : MEGxp
  Amount_group_85_100_percent   : 100
  Amount_group_70_85_percent    : 100
  Novelty                       : N
  TPSA                          : 65.12
  ACC                           : 3
  DON                           : 3
  ROT                           : 4
  clogP                         : -0.862
  FSP3                          : 0.25
  similar structure             : 
  formula                       : C12H14N2O2
  molweight                     : 218.256
  chemical class                : Aminoacids and peptides
  largest ring                  : 6
  SMILES                        : CN[C@@H](CC1=CNC2=CC=CC=C12)C(O)=O
  InChIKey                      : InChIKey=CZCIKBSVHDNIDH-LDGXTIHJNA-N

SMILES: CN[C@@H](Cc1c[nH]c2ccccc12)C(=O)

## Screen MEGx Library

In [6]:
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
from rdkit.Chem import rdMolDescriptors
import pandas as pd
import numpy as np
import time
from collections import Counter

sdf_path = r"D:\JK\CheB\Remaining Libraries to run\MEGx_Release_2025_09_03_All_6539_cpds.sdf"

# ZBG patterns
zbg_patterns = {
    'Hydroxamic acid' : Chem.MolFromSmarts('[CX3](=O)[NX3][OH]'),
    'Carboxylic acid' : Chem.MolFromSmarts('[CX3](=O)[OH]'),
    'Thiol'           : Chem.MolFromSmarts('[SX2H]'),
    'Phosphonate'     : Chem.MolFromSmarts('[PX4](=O)([OH])[OH]'),
    'Sulfonamide'     : Chem.MolFromSmarts('[SX4](=O)(=O)[NX3]'),
    'Catechol'        : Chem.MolFromSmarts('c1ccc(O)c(O)c1'),
    'N-hydroxy'       : Chem.MolFromSmarts('[NX3][OH]'),
    'Hydroxypyridine' : Chem.MolFromSmarts('n1ccccc1O'),
    'Beta-lactam'     : Chem.MolFromSmarts('[NX3]1[CX4][CX3](=O)1'),
}

print("Step 1 — Filtering MEGx library...")
supplier = Chem.SDMolSupplier(sdf_path)

filtered_data = []
t0 = time.time()

for i, mol in enumerate(supplier):
    if mol is None:
        continue
    try:
        smiles = Chem.MolToSmiles(mol)
        props  = mol.GetPropsAsDict()

        comp_id  = props.get('id', f'IBS_{i}')
        comment  = props.get('comment', '')
        iupac    = props.get('iupac_name', '')
        mw       = Descriptors.MolWt(mol)
        logp     = float(props.get('clogp',
                         Descriptors.MolLogP(mol)))
        tpsa     = float(props.get('tpsa', 0))
        hbd      = int(props.get('donors',
                       rdMolDescriptors.CalcNumHBD(mol)))
        hba      = int(props.get('acceptors',
                       rdMolDescriptors.CalcNumHBA(mol)))
        rings    = int(props.get('rings', 0))

        # ZBG filter
        found_zbgs = []
        for zbg_name, pattern in zbg_patterns.items():
            if pattern and mol.HasSubstructMatch(pattern):
                found_zbgs.append(zbg_name)

        if not found_zbgs:
            continue

        filtered_data.append({
            'id'     : comp_id,
            'iupac'  : iupac,
            'comment': comment,
            'smiles' : smiles,
            'MW'     : round(mw, 2),
            'LogP'   : round(logp, 2),
            'TPSA'   : round(tpsa, 2),
            'HBD'    : hbd,
            'HBA'    : hba,
            'rings'  : rings,
            'ZBG'    : ', '.join(found_zbgs)
        })
    except:
        continue

    if (i+1) % 10000 == 0:
        elapsed = round((time.time()-t0)/60, 1)
        print(f"  Processed: {i+1:,} | "
              f"ZBG pass: {len(filtered_data):,} | "
              f"Time: {elapsed} mins")

filtered_df = pd.DataFrame(filtered_data)
print(f"\nTotal processed : 68,988")
print(f"ZBG passed      : {len(filtered_df):,}")
print(f"\nZBG breakdown:")
all_zbgs = []
for zbg in filtered_df['ZBG']:
    all_zbgs.extend(zbg.split(', '))
for zbg, count in Counter(all_zbgs).most_common():
    print(f"  {zbg:25s}: {count:,}")

# Step 2 — Screen with RF model
print("\nStep 2 — Screening with RF model...")
descriptor_names = [d[0] for d in Descriptors.descList]

desc_list, fp_list, valid_idx = [], [], []

for i, smiles in enumerate(filtered_df['smiles']):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is not None:
        descs = calculator.CalcDescriptors(mol)
        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol, radius=2, nBits=2048)
        desc_list.append(descs)
        fp_list.append(list(fp))
        valid_idx.append(i)
    if (i+1) % 2000 == 0:
        print(f"  {i+1:,}/{len(filtered_df):,}...",
              end='\r')

desc_df = pd.DataFrame(desc_list, columns=descriptor_names)
fp_df   = pd.DataFrame(
    fp_list,
    columns=[f'Morgan_{i}' for i in range(2048)]
)

X_desc = desc_df[clean_descriptors].copy()
X_desc = X_desc.replace([np.inf, -np.inf], np.nan)
X_desc = X_desc.fillna(X_desc.median())
X_desc = X_desc.clip(
    -np.finfo(np.float32).max,
     np.finfo(np.float32).max
).astype(np.float32)

X_comb = pd.concat([
    X_desc.reset_index(drop=True),
    fp_df.reset_index(drop=True)
], axis=1)
X_comb = X_comb.replace([np.inf, -np.inf], np.nan)
X_comb = X_comb.fillna(X_comb.median())
X_comb = X_comb.clip(
    -np.finfo(np.float32).max,
     np.finfo(np.float32).max
).astype(np.float32)

activity_pred = rf_model.predict(X_desc)
activity_prob = rf_model.predict_proba(X_desc)[:, 1]
pic50_pred    = rf_reg.predict(X_comb)

results = filtered_df.iloc[
    valid_idx
].reset_index(drop=True).copy()
results['predicted_activity'] = activity_pred
results['active_probability'] = activity_prob.round(4)
results['predicted_pIC50']    = pic50_pred.round(3)

actives = results[
    results['predicted_activity'] == 1
].sort_values(
    'active_probability', ascending=False
).reset_index(drop=True)

high_conf = actives[
    actives['active_probability'] >= 0.85
].copy()

# Save
actives.to_csv(
    project_path + r'\Megx_all_hits.csv', index=False
)
high_conf.to_csv(
    project_path + r'\glide_docking\megx_docking.csv',
    index=False
)

elapsed = round((time.time()-t0)/60, 2)
print(f"\n{'='*55}")
print(f"IBS SCREENING COMPLETE")
print(f"{'='*55}")
print(f"ZBG compounds    : {len(filtered_df):,}")
print(f"Valid molecules  : {len(valid_idx):,}")
print(f"Actives          : {len(actives):,}")
print(f"High conf (≥0.85): {len(high_conf):,}")
print(f"Time             : {elapsed} mins")
print(f"\nTop 20 High Confidence Hits:")
print(high_conf.head(20)[[
    'id', 'ZBG', 'MW', 'LogP',
    'active_probability', 'predicted_pIC50'
]].to_string(index=False))

Step 1 — Filtering MEGx library...


[15:07:22] The 2 defining bonds for an atropisomer are co-planar - atoms are: 19 21
[15:07:22] Both bonds on one end of an atropisomer are on the same side - atoms is : 8
[15:07:23] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 9 ignored.
[15:07:23] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 9 ignored.
[15:07:24] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 3 ignored.
[15:07:24] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 3 ignored.
[15:07:25] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 24 ignored
[15:07:25] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 24 ignored
[15:07:25] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 14 ignored
[15:07:25] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 14 ignored
[15:07:25] Wa


Total processed : 68,988
ZBG passed      : 2,887

ZBG breakdown:
  Carboxylic acid          : 1,837
  Catechol                 : 1,221
  N-hydroxy                : 24
  Hydroxamic acid          : 19
  Thiol                    : 2

Step 2 — Screening with RF model...


[15:07:37] DEPRECATION WARNING: please use MorganGenerator
[15:07:37] DEPRECATION WARNING: please use MorganGenerator
[15:07:37] DEPRECATION WARNING: please use MorganGenerator
[15:07:37] DEPRECATION WARNING: please use MorganGenerator
[15:07:37] DEPRECATION WARNING: please use MorganGenerator
[15:07:37] DEPRECATION WARNING: please use MorganGenerator
[15:07:37] DEPRECATION WARNING: please use MorganGenerator
[15:07:37] DEPRECATION WARNING: please use MorganGenerator
[15:07:37] DEPRECATION WARNING: please use MorganGenerator
[15:07:37] DEPRECATION WARNING: please use MorganGenerator
[15:07:37] DEPRECATION WARNING: please use MorganGenerator
[15:07:37] DEPRECATION WARNING: please use MorganGenerator
[15:07:37] DEPRECATION WARNING: please use MorganGenerator
[15:07:37] DEPRECATION WARNING: please use MorganGenerator
[15:07:37] DEPRECATION WARNING: please use MorganGenerator
[15:07:37] DEPRECATION WARNING: please use MorganGenerator
[15:07:37] DEPRECATION WARNING: please use MorganGenerat

  2,000/2,887...

[15:08:29] DEPRECATION WARNING: please use MorganGenerator
[15:08:29] DEPRECATION WARNING: please use MorganGenerator
[15:08:29] DEPRECATION WARNING: please use MorganGenerator
[15:08:29] DEPRECATION WARNING: please use MorganGenerator
[15:08:29] DEPRECATION WARNING: please use MorganGenerator
[15:08:29] DEPRECATION WARNING: please use MorganGenerator
[15:08:29] DEPRECATION WARNING: please use MorganGenerator
[15:08:29] DEPRECATION WARNING: please use MorganGenerator
[15:08:30] DEPRECATION WARNING: please use MorganGenerator
[15:08:30] DEPRECATION WARNING: please use MorganGenerator
[15:08:30] DEPRECATION WARNING: please use MorganGenerator
[15:08:30] DEPRECATION WARNING: please use MorganGenerator
[15:08:30] DEPRECATION WARNING: please use MorganGenerator
[15:08:30] DEPRECATION WARNING: please use MorganGenerator
[15:08:30] DEPRECATION WARNING: please use MorganGenerator
[15:08:30] DEPRECATION WARNING: please use MorganGenerator
[15:08:30] DEPRECATION WARNING: please use MorganGenerat


IBS SCREENING COMPLETE
ZBG compounds    : 2,887
Valid molecules  : 2,887
Actives          : 2,410
High conf (≥0.85): 127
Time             : 1.61 mins

Top 20 High Confidence Hits:
      id                       ZBG     MW  LogP  active_probability  predicted_pIC50
IBS_2759                  Catechol 476.48  0.33                0.92            8.207
IBS_3484 Carboxylic acid, Catechol 562.48  0.56                0.92            8.047
IBS_2875 Carboxylic acid, Catechol 432.38  1.17                0.91            8.111
 IBS_224 Carboxylic acid, Catechol 448.38  0.88                0.91            8.091
IBS_2452 Carboxylic acid, Catechol 432.38  1.17                0.91            8.155
IBS_5481 Carboxylic acid, Catechol 616.53  1.45                0.91            8.317
IBS_5377           Carboxylic acid 822.77 -0.01                0.91            8.210
  IBS_43 Carboxylic acid, Catechol 516.46  1.03                0.90            8.188
IBS_3175 Carboxylic acid, Catechol 560.51  1.34       

## MEGx SDF Hits file

In [7]:
from rdkit import Chem
from rdkit.Chem import AllChem
import pandas as pd

# Load high confidence hits
high_conf = pd.read_csv(
    project_path + r'\glide_docking\megx_docking.csv'
)

print(f"Total compounds to convert: {len(high_conf)}")

# Write SDF
sdf_path = project_path + \
    r'\glide_docking\megx_highconf_935.sdf'

writer = Chem.SDWriter(sdf_path)
success = 0
failed  = 0

for _, row in high_conf.iterrows():
    try:
        mol = Chem.MolFromSmiles(str(row['smiles']))
        if mol is None:
            failed += 1
            continue

        # Add hydrogens and generate 3D
        mol = Chem.AddHs(mol)
        result = AllChem.EmbedMolecule(
            mol, AllChem.ETKDGv3()
        )

        if result == 0:
            AllChem.MMFFOptimizeMolecule(mol)
        else:
            # Fall back to 2D if 3D fails
            mol = Chem.RemoveHs(mol)
            AllChem.Compute2DCoords(mol)

        # Set properties
        mol.SetProp('_Name',            str(row['id']))
        mol.SetProp('ZBG',              str(row['ZBG']))
        mol.SetProp('MW',               str(row['MW']))
        mol.SetProp('LogP',             str(row['LogP']))
        mol.SetProp('active_probability',
                    str(row['active_probability']))
        mol.SetProp('predicted_pIC50',
                    str(row['predicted_pIC50']))

        writer.write(mol)
        success += 1

        if success % 100 == 0:
            print(f"  Written: {success}...", end='\r')

    except:
        failed += 1

writer.close()

print(f"\n{'='*50}")
print(f"SDF FILE CREATED ✅")
print(f"{'='*50}")
print(f"Successfully written : {success}")
print(f"Failed               : {failed}")
print(f"Saved to             : {sdf_path}")

Total compounds to convert: 127
  Written: 100...
SDF FILE CREATED ✅
Successfully written : 127
Failed               : 0
Saved to             : C:\Users\Dell\Documents\SK\MyProject\Task1\glide_docking\megx_highconf_935.sdf


## Asinex Screening

In [8]:
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
from rdkit.Chem import rdMolDescriptors
import pandas as pd
import numpy as np
import time
from collections import Counter

sdf_path = r"C:\Users\DELL\Documents\SK\MyProject\Task1\glide_docking\2021-04 Asinex Macrocycles - 10091.sdf"

# ZBG patterns
zbg_patterns = {
    'Hydroxamic acid' : Chem.MolFromSmarts('[CX3](=O)[NX3][OH]'),
    'Carboxylic acid' : Chem.MolFromSmarts('[CX3](=O)[OH]'),
    'Thiol'           : Chem.MolFromSmarts('[SX2H]'),
    'Phosphonate'     : Chem.MolFromSmarts('[PX4](=O)([OH])[OH]'),
    'Sulfonamide'     : Chem.MolFromSmarts('[SX4](=O)(=O)[NX3]'),
    'Catechol'        : Chem.MolFromSmarts('c1ccc(O)c(O)c1'),
    'N-hydroxy'       : Chem.MolFromSmarts('[NX3][OH]'),
    'Hydroxypyridine' : Chem.MolFromSmarts('n1ccccc1O'),
    'Beta-lactam'     : Chem.MolFromSmarts('[NX3]1[CX4][CX3](=O)1'),
}

print("Step 1 — Filtering Asinex library...")
supplier = Chem.SDMolSupplier(sdf_path)

filtered_data = []
t0 = time.time()

for i, mol in enumerate(supplier):
    if mol is None:
        continue
    try:
        smiles = Chem.MolToSmiles(mol)
        props  = mol.GetPropsAsDict()

        comp_id  = props.get('id', f'IBS_{i}')
        comment  = props.get('comment', '')
        iupac    = props.get('iupac_name', '')
        mw       = Descriptors.MolWt(mol)
        logp     = float(props.get('clogp',
                         Descriptors.MolLogP(mol)))
        tpsa     = float(props.get('tpsa', 0))
        hbd      = int(props.get('donors',
                       rdMolDescriptors.CalcNumHBD(mol)))
        hba      = int(props.get('acceptors',
                       rdMolDescriptors.CalcNumHBA(mol)))
        rings    = int(props.get('rings', 0))

        # ZBG filter
        found_zbgs = []
        for zbg_name, pattern in zbg_patterns.items():
            if pattern and mol.HasSubstructMatch(pattern):
                found_zbgs.append(zbg_name)

        if not found_zbgs:
            continue

        filtered_data.append({
            'id'     : comp_id,
            'iupac'  : iupac,
            'comment': comment,
            'smiles' : smiles,
            'MW'     : round(mw, 2),
            'LogP'   : round(logp, 2),
            'TPSA'   : round(tpsa, 2),
            'HBD'    : hbd,
            'HBA'    : hba,
            'rings'  : rings,
            'ZBG'    : ', '.join(found_zbgs)
        })
    except:
        continue

    if (i+1) % 10000 == 0:
        elapsed = round((time.time()-t0)/60, 1)
        print(f"  Processed: {i+1:,} | "
              f"ZBG pass: {len(filtered_data):,} | "
              f"Time: {elapsed} mins")

filtered_df = pd.DataFrame(filtered_data)
print(f"\nTotal processed : 68,988")
print(f"ZBG passed      : {len(filtered_df):,}")
print(f"\nZBG breakdown:")
all_zbgs = []
for zbg in filtered_df['ZBG']:
    all_zbgs.extend(zbg.split(', '))
for zbg, count in Counter(all_zbgs).most_common():
    print(f"  {zbg:25s}: {count:,}")

# Step 2 — Screen with RF model
print("\nStep 2 — Screening with RF model...")
descriptor_names = [d[0] for d in Descriptors.descList]

desc_list, fp_list, valid_idx = [], [], []

for i, smiles in enumerate(filtered_df['smiles']):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is not None:
        descs = calculator.CalcDescriptors(mol)
        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol, radius=2, nBits=2048)
        desc_list.append(descs)
        fp_list.append(list(fp))
        valid_idx.append(i)
    if (i+1) % 2000 == 0:
        print(f"  {i+1:,}/{len(filtered_df):,}...",
              end='\r')

desc_df = pd.DataFrame(desc_list, columns=descriptor_names)
fp_df   = pd.DataFrame(
    fp_list,
    columns=[f'Morgan_{i}' for i in range(2048)]
)

X_desc = desc_df[clean_descriptors].copy()
X_desc = X_desc.replace([np.inf, -np.inf], np.nan)
X_desc = X_desc.fillna(X_desc.median())
X_desc = X_desc.clip(
    -np.finfo(np.float32).max,
     np.finfo(np.float32).max
).astype(np.float32)

X_comb = pd.concat([
    X_desc.reset_index(drop=True),
    fp_df.reset_index(drop=True)
], axis=1)
X_comb = X_comb.replace([np.inf, -np.inf], np.nan)
X_comb = X_comb.fillna(X_comb.median())
X_comb = X_comb.clip(
    -np.finfo(np.float32).max,
     np.finfo(np.float32).max
).astype(np.float32)

activity_pred = rf_model.predict(X_desc)
activity_prob = rf_model.predict_proba(X_desc)[:, 1]
pic50_pred    = rf_reg.predict(X_comb)

results = filtered_df.iloc[
    valid_idx
].reset_index(drop=True).copy()
results['predicted_activity'] = activity_pred
results['active_probability'] = activity_prob.round(4)
results['predicted_pIC50']    = pic50_pred.round(3)

actives = results[
    results['predicted_activity'] == 1
].sort_values(
    'active_probability', ascending=False
).reset_index(drop=True)

high_conf = actives[
    actives['active_probability'] >= 0.85
].copy()

# Save
actives.to_csv(
    project_path + r'\asinex_all_hits.csv', index=False
)
high_conf.to_csv(
    project_path + r'\glide_docking\asinex_docking.csv',
    index=False
)

elapsed = round((time.time()-t0)/60, 2)
print(f"\n{'='*55}")
print(f"ASINEX SCREENING COMPLETE")
print(f"{'='*55}")
print(f"ZBG compounds    : {len(filtered_df):,}")
print(f"Valid molecules  : {len(valid_idx):,}")
print(f"Actives          : {len(actives):,}")
print(f"High conf (≥0.85): {len(high_conf):,}")
print(f"Time             : {elapsed} mins")
print(f"\nTop 20 High Confidence Hits:")
print(high_conf.head(20)[[
    'id', 'ZBG', 'MW', 'LogP',
    'active_probability', 'predicted_pIC50'
]].to_string(index=False))

Step 1 — Filtering Asinex library...

Total processed : 68,988
ZBG passed      : 1,376

ZBG breakdown:
  Catechol                 : 687
  Sulfonamide              : 353
  Carboxylic acid          : 301
  Hydroxypyridine          : 230

Step 2 — Screening with RF model...


[15:47:53] DEPRECATION WARNING: please use MorganGenerator
[15:47:53] DEPRECATION WARNING: please use MorganGenerator
[15:47:53] DEPRECATION WARNING: please use MorganGenerator
[15:47:53] DEPRECATION WARNING: please use MorganGenerator
[15:47:53] DEPRECATION WARNING: please use MorganGenerator
[15:47:53] DEPRECATION WARNING: please use MorganGenerator
[15:47:53] DEPRECATION WARNING: please use MorganGenerator
[15:47:53] DEPRECATION WARNING: please use MorganGenerator
[15:47:53] DEPRECATION WARNING: please use MorganGenerator
[15:47:53] DEPRECATION WARNING: please use MorganGenerator
[15:47:53] DEPRECATION WARNING: please use MorganGenerator
[15:47:53] DEPRECATION WARNING: please use MorganGenerator
[15:47:53] DEPRECATION WARNING: please use MorganGenerator
[15:47:53] DEPRECATION WARNING: please use MorganGenerator
[15:47:53] DEPRECATION WARNING: please use MorganGenerator
[15:47:53] DEPRECATION WARNING: please use MorganGenerator
[15:47:54] DEPRECATION WARNING: please use MorganGenerat


ASINEX SCREENING COMPLETE
ZBG compounds    : 1,376
Valid molecules  : 1,376
Actives          : 1,233
High conf (≥0.85): 56
Time             : 1.03 mins

Top 20 High Confidence Hits:
      id             ZBG     MW  LogP  active_probability  predicted_pIC50
IBS_2200 Carboxylic acid 469.49  0.93                0.92            8.095
IBS_4389 Carboxylic acid 469.49  0.98                0.91            6.637
 IBS_792     Sulfonamide 527.60  1.35                0.91            8.398
IBS_4545 Carboxylic acid 502.57  0.73                0.91            8.093
IBS_6207 Carboxylic acid 602.64  1.32                0.91            8.239
IBS_4619 Carboxylic acid 469.49  0.25                0.90            8.089
IBS_6209 Carboxylic acid 540.57 -0.25                0.90            8.101
 IBS_737 Carboxylic acid 487.94  1.88                0.90            8.247
IBS_4206 Carboxylic acid 446.46 -0.61                0.90            8.173
IBS_5125 Carboxylic acid 445.52  1.61                0.90          

## Asinex Sdf FileCreation

In [9]:
from rdkit import Chem
from rdkit.Chem import AllChem
import pandas as pd

# Load high confidence hits
high_conf = pd.read_csv(
    project_path + r'\glide_docking\asinex_docking.csv'
)

print(f"Total compounds to convert: {len(high_conf)}")

# Write SDF
sdf_path = project_path + \
    r'\glide_docking\asinex_highconf_935.sdf'

writer = Chem.SDWriter(sdf_path)
success = 0
failed  = 0

for _, row in high_conf.iterrows():
    try:
        mol = Chem.MolFromSmiles(str(row['smiles']))
        if mol is None:
            failed += 1
            continue

        # Add hydrogens and generate 3D
        mol = Chem.AddHs(mol)
        result = AllChem.EmbedMolecule(
            mol, AllChem.ETKDGv3()
        )

        if result == 0:
            AllChem.MMFFOptimizeMolecule(mol)
        else:
            # Fall back to 2D if 3D fails
            mol = Chem.RemoveHs(mol)
            AllChem.Compute2DCoords(mol)

        # Set properties
        mol.SetProp('_Name',            str(row['id']))
        mol.SetProp('ZBG',              str(row['ZBG']))
        mol.SetProp('MW',               str(row['MW']))
        mol.SetProp('LogP',             str(row['LogP']))
        mol.SetProp('active_probability',
                    str(row['active_probability']))
        mol.SetProp('predicted_pIC50',
                    str(row['predicted_pIC50']))

        writer.write(mol)
        success += 1

        if success % 100 == 0:
            print(f"  Written: {success}...", end='\r')

    except:
        failed += 1

writer.close()

print(f"\n{'='*50}")
print(f"SDF FILE CREATED ✅")
print(f"{'='*50}")
print(f"Successfully written : {success}")
print(f"Failed               : {failed}")
print(f"Saved to             : {sdf_path}")

Total compounds to convert: 56

SDF FILE CREATED ✅
Successfully written : 56
Failed               : 0
Saved to             : C:\Users\Dell\Documents\SK\MyProject\Task1\glide_docking\asinex_highconf_935.sdf
